### The following code mainly concerns the sixteen meta-programs and the top 10 genes with the highest weights in each module, as determined by GeneNMF (Supplementary Fig. 11)

In [ ]:
library(GeneNMF)
library(Seurat)
library(ggplot2)
library(UCell)
library(patchwork)
library(Matrix)
library(RcppML)
library(viridis)
library(qs)
library(dplyr)
library(stringr)

In [ ]:
setwd('/mnt/data/khm_scRNA/huh7_zxs/result_zxs/')

In [ ]:
seurat_obj <- readRDS('/mnt/data/khm_scRNA/huh7_zxs/result_zxs/scdata_filter.rds')
seurat_obj

In [ ]:
seurat_obj$batch %>% unique()

In [ ]:
seurat_obj <- subset(seurat_obj,subset = batch %in% c('normal2','Tatin'))

In [ ]:
seurat_obj$batch %>% table()

In [ ]:
seu <- seurat_obj
ndim <- 15

In [ ]:
length(rownames(seurat_obj))

In [ ]:
seu <- FindVariableFeatures(seu, nfeatures = 3000) 
seu <- runNMF(seu, k = ndim, assay="RNA") 
seu@reductions$NMF

In [ ]:
seu <- RunUMAP(
    seu, 
    reduction = "NMF", 
    dims=1:ndim, 
    reduction.name = "NMF_UMAP", 
    reduction.key = "nmfUMAP_")

In [ ]:
library(dplyr, help, pos = 2, lib.loc = NULL)
seu@meta.data %>% head()

### 不同MP的top 10Gene的柱状图

In [ ]:
data_module <- lapply(names(geneNMF.metaprograms$metaprograms.genes.weights),function(Module_use){
    df <- geneNMF.metaprograms$metaprograms.genes.weights[[Module_use]] %>% as.data.frame() %>% 
        tibble::rownames_to_column('Genes') %>% 
        rename_all(~c('Genes','Module_score')) %>% 
        mutate(Module = Module_use)
    return(df)
}) %>% do.call(rbind,.) %>% 
    group_by(Module) %>% 
    arrange(desc(Module_score),.by_group = TRUE) %>% 
    slice(1:10) %>% 
    mutate(Module = factor(Module,levels = paste('MP',1:16,sep = '')))
data_module %>% head()

In [ ]:
write.csv(data_module,'../result_figs/data_module_Genes_top10.csv')

In [ ]:
data_module$Module %>% levels()

In [ ]:
p_list <- lapply(data_module$Module %>% levels(),function(Module_select){
    data_plot <- data_module %>% 
        filter(Module == Module_select) %>% 
        arrange(Module_score) %>% 
        mutate(Genes = factor(Genes,levels = Genes %>% unique()))
    options(repr.plot.height = 5,repr.plot.width = 6)
    p <- ggplot(data_plot, aes(x = Module_score, y = Genes)) + #, fill = -log10(p.adjust)
      geom_bar(stat = 'identity',width = 0.8,fill = '#5D8BBA') +
      guides(fill = guide_colorbar(title.position = "left", title.theme = element_text(angle = 90,vjust = 0.5))) +
      scale_x_continuous(expand = c(0.01,0)) +
      scale_y_discrete(expand = c(0,1)) +
      labs(y = 'Genes',x = 'Weights',title = paste('Top 10 core Genes of Module ',Module_select,sep = '')) +
      theme_classic(base_size = 20) +
      coord_cartesian(clip = "off") +
      theme(
        plot.margin = unit(c(60, 0, 0, 0), "pt"),
        plot.title = element_text(hjust = 0,size = 16),
        # legend.position = 'none',
        legend.title = element_text(size = 24,hjust = 0),
        legend.text = element_text(size = 20),
        legend.key.height = unit(1.4, "cm"),
        legend.key.width = unit(1.2, "cm"),
        # axis.text.x = element_text(size = 20,angle = 90,hjust = 1,vjust = 0.5),
        axis.text.x = element_blank(),
        # axis.text.y = element_blank(),
        # axis.title.x = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
        strip.placement = 'outside'
      )
    return(p)
})